# Determine The Position Of Earth With Respect To The Sun

In [15]:
import spiceypy
import datetime
import math
from numpy.core._multiarray_umath import ndarray
from typing import Union

## Auxiliary Functions

In [16]:
def get_todays_et() -> float:
    """
    Convert and return todays (midnight) datetime to ET (Ephemeris Time) with the help of
    SPICE's `utc2et`_ function

    Example:
        2023-03-05T00:00:00 to 731246469.1854347

    .. _utc2et: https://naif.jpl.nasa.gov/pub/naif/toolkit_docs/C/cspice/utc2et_c.html
    :rtype: float
    :return: Todays datetime converted into ET format
    """
    date_today = datetime.datetime.now()
    date_today = date_today.strftime('%Y-%m-%dT00:00:00')

    spiceypy.furnsh('../../kernels/lsk/naif0012.tls')
    return spiceypy.utc2et(date_today)

In [17]:
def get_earth_state_vector(tar_body: int, tar_epoch: float, tar_ref_frame: str, obs_body: int) -> tuple[ndarray, float]:
    """
    Computes the Earth's geometric state vector with respect to the Sun by the help of SPICE's `spkgeo`_ function

    The targets state vector contains position information (first 3 values [x, y and z] in km) and velocity information (last 3 values [x, y and z] in km/s). To
    determine the target (Earth) and observer (Sun), we need to check the respectic `NAIF IDs`_ used by SPICE

    Example:
        * [-1.42466905e+08  4.13241191e+07 -1.80206727e+03 -8.77453905e+00 -2.87142917e+01  1.75358712e-03] as **state of target**
        * 494.80612435799 as **light time**

    This can be verified by using `NASAs HORIZONS Web-Interface`_

    .. _spkgeo: https://naif.jpl.nasa.gov/pub/naif/toolkit_docs/C/cspice/spkgeo_c.html
    .. _NAIF IDs: https://naif.jpl.nasa.gov/pub/naif/toolkit_docs/C/req/naif_ids.html
    .. _NASAs HORIZONS Web-Interface: https://ssd.jpl.nasa.gov/horizons/app.html#/
    :rtype: tuple[ndarray, float]
    :param tar_body: Target body
    :param tar_epoch: Target epoch
    :param tar_ref_frame: Target reference frame
    :param obs_body: Observing body
    :return: The computed Earth state vector and its light time
    """
    spiceypy.furnsh('../../kernels/spk/de432s.bsp')  # Determined with help of https://naif.jpl.nasa.gov/pub/naif/generic_kernels/spk/planets/aa_summaries.txt
    return spiceypy.spkgeo(targ=tar_body, et=tar_epoch, ref=tar_ref_frame, obs=obs_body)

In [18]:
def determine_distance_earth_sun(earth_state: ndarray) -> Union[ndarray, float]:
    """
    Determines the distance between the Earth and the Sun by computing it in **km** with the help of the Earths state vector and
    then converting it into **AU** with the help of SPICE's `convrt`_ function

    The (euclidean) distance between Earth and Sun should be around 1 AU, since Earth revolves the Sun in a slightly non-perfect circle (elliptical orbit)

    .. _convrt: https://naif.jpl.nasa.gov/pub/naif/toolkit_docs/C/cspice/convrt_c.html
    :rtype: Union[ndarray, float]
    :param earth_state: The Earth state vector w.r.t the Sun
    :return: The distance in **AU** unit
    """
    distance = math.sqrt(earth_state[0] ** 2.0 + earth_state[1] ** 2.0 + earth_state[2] ** 2.0)
    return spiceypy.convrt(distance, 'km', 'AU')

In [19]:
def determine_orbital_speed_of_earth(earth_state: ndarray, obs_body: int) -> tuple[float, float]:
    """
    Determines the orbital speed of the Earth w.r.t the Sun in **km/s** and the theoretical expectation velocity with the help
    of SPICE's `bodvcd`_ function

    .. _bodvcd: https://naif.jpl.nasa.gov/pub/naif/toolkit_docs/C/cspice/bodvcd_c.html
    :rtype: tuple[float, float]
    :param earth_state: The Earth state vector w.r.t the Sun
    :param obs_body: Observing body
    :return: Orbital speed of the Earth and theoretical orbital speed in **km/s**
    """
    spiceypy.furnsh('../../kernels/pck/gm_de431.tpc')
    _, gm_sun = spiceypy.bodvcd(bodyid=obs_body, item='GM', maxn=1)

    distance = math.sqrt(earth_state[0] ** 2.0 + earth_state[1] ** 2.0 + earth_state[2] ** 2.0)
    theoretical_orbital_speed = orbital_speed_equation(gm_sun[0], distance)

    return math.sqrt(earth_state[3]**2.0 + earth_state[4]**2.0 + earth_state[5]**2.0), theoretical_orbital_speed

In [20]:
def orbital_speed_equation(gm: float, r: float) -> float:
    """
    .. math::
        v = \sqrt{GM / r}
    ..

     Where the `Orbital Speed Equation`_ consists of:
        * v = Velocity
        * G = Universal Gravitational Constant
        * M = Mass of Planet
        * r = Distance between point and the planet

    .. _Orbital Speed Equation: https://www.nagwa.com/en/explainers/142168516704/
    :rtype: float
    :param gm: Universal Gravitational Constant * Mass of Planet
    :param r: Distance between point and the planet
    :return: Velocity
    """
    return math.sqrt(gm / r)

In [21]:
earth_naif_id = 399
sun_naif_id = 10
et_today = get_todays_et()

## Compute State Vector

In [22]:
earth_state_wrt_sun, earth_sun_lt = get_earth_state_vector(earth_naif_id, et_today, 'ECLIPJ2000', sun_naif_id)
print('Earth State Vector: ', earth_state_wrt_sun)
print('Earth to Sun Light Time: ', earth_sun_lt)

Earth State Vector:  [-1.42466905e+08  4.13241191e+07 -1.80206727e+03 -8.77453905e+00
 -2.87142917e+01  1.75358712e-03]
Earth to Sun Light Time:  494.80612435799


## Determine Distance Between Earth And Sun

In [23]:
earth_sun_distance = determine_distance_earth_sun(earth_state_wrt_sun)
print('Distance between Earth and Sun in AU: ', earth_sun_distance)

Distance between Earth and Sun in AU:  0.9915859339856259


## Determine Orbital Speed Of Earth

For this, we need the equation to determine the orbital speed. We assume that the Sun's mass is greater than the mass
of the Earth and we assume that our planet is moving on an almost circular orbit. The orbit velocity $v_{\text{orb}}$ can be approximated with, where $G$ is the gravitational constant, $M$ is the mass of the Sun and $r$ is the distance between the Earth and the Sun:
\begin{align}
    v_{\text{orb}}\approx\sqrt{\frac{GM}{r}}
\end{align}

In [24]:
earth_orb_speed_wrt_sun, earth_orb_speed_wrt_sun_theory = determine_orbital_speed_of_earth(earth_state_wrt_sun, sun_naif_id)
print('Current orbital speed of the Earth around the Sun in km/s: ', earth_orb_speed_wrt_sun)
print('Theoretical orbital speed of the Earth around the Sun in km/s: ', earth_orb_speed_wrt_sun_theory)

Current orbital speed of the Earth around the Sun in km/s:  30.02504103612176
Theoretical orbital speed of the Earth around the Sun in km/s:  29.910793354833096
